# Algebra & Multivectors — The Core

**Part II · Geometric Algebra** — Tutorial 02

This tutorial introduces the two objects at the heart of *pytanga*: the **`Algebra`**
class and the **`MV`** (multivector) type. Everything else in the library — the basis
classes, the geometry submodule, the numerical tooling — is built on top of these two.

By the end you will be able to:

- Construct a geometric algebra from its **dimension**, **signature**, and **dtype**.
- Create multivectors from **strings, dictionaries, and tuples**.
- Read and write individual **blade coefficients**.
- Compute the **geometric, outer, and inner** products.
- Apply the **reverse**, **involutions**, and the **inverse**.
- Extract **grades**, iterate over **blades**, and clean up results with **`prune()`**.


## 1. Setup

Only the two core types are needed. They are re-exported from the top-level `pytanga`
package, so this is the canonical import:


In [1]:
from pytanga import Algebra, MV


## 2. The `Algebra` class

An `Algebra` instance represents a **geometric algebra** $G(d, s)$ — a Clifford algebra
built over a real vector space of dimension $d$, with a quadratic form whose *signature*
is encoded by $s$.

- **dimension** $d$ — how many basis vectors $e_1 \dots e_d$ span the space.
- **signature** $s$ — which basis vectors square to $-1$ (all others square to $+1$).
- **dtype** — the numeric type of the coefficients (`float64` is the default).

From the $d$ basis vectors, all possible products form the *blades* of the algebra.
There are $2^d$ blades in total — in 3D that is the 8 blades
$s,\ e_1,\ e_2,\ e_3,\ e_{12},\ e_{13},\ e_{23},\ e_{123}$.

> **Compilation note.** The first time a new `(d, s, dtype)` combination is used, the
> C++ binding is compiled and cached (~5–20 s). Precompiled wheels cover the standard
> configurations (e.g. G(3,0) `float64`/`int64`), so those load instantly. A compiler is
> only needed for configurations outside the precompiled set.

Let's create the simplest non-trivial algebra — 3D Euclidean space $G(3, 0)$:


In [2]:
alg = Algebra(3, 0)  # G(3, 0): 3 basis vectors, all square to +1

print("dim          :", alg.dim)
print("sig          :", alg.sig)
print("dtype        :", alg.dtype)
print("algebra_dim  :", alg.algebra_dim)     # 2**3 = 8 blades
print("pseudoscalar :", alg.pseudoscalar_id)  # bitmask of the pseudoscalar I


dim          : 3
sig          : 0
dtype        : float64
algebra_dim  : 8
pseudoscalar : 7


### The signature

The signature is a **bitmask**: bit $k$ set means basis vector $e_{k+1}$ squares to
$-1$. You may also pass a tuple of 1-based indices for readability — `(4,)` and the
bitmask `0b1000` (bit 3 set) both mean "only $e_4$ squares to $-1$":


In [3]:
alg4a = Algebra(4, (4,))    # G(4, e4^2 = -1)  via tuple
alg4b = Algebra(4, 0b1000)  # same algebra, via raw bitmask

print("tuple form  sig =", alg4a.sig)
print("bitmask form sig =", alg4b.sig)
print("same signature  :", alg4a.sig == alg4b.sig)

# Confirm the metric: e4 squares to -1, e1 squares to +1
g1 = alg4b("e1")
g4 = alg4b("e4")
print("e1 * e1 =", g1 * g1)
print("e4 * e4 =", g4 * g4)


tuple form  sig = 8
bitmask form sig = 8
same signature  : True
e1 * e1 = 1
e4 * e4 = -1


### The dtype

Coefficients can be stored as `float64` (default), `float32`, `int64`, or `int32`.
Integer dtypes give exact arithmetic and are the basis of the modular-algebra features
(covered in a later tutorial):


In [4]:
alg_f = Algebra(3, 0)            # float64 (default)
alg_i = Algebra(3, 0, "int64")   # exact integer coefficients

v_f = alg_f("e1 + 2 e2")
v_i = alg_i("e1 + 2 e2")

print("float64:", v_f, "| coefficient type:", type(v_f["e1"]).__name__)
print("int64  :", v_i, "| coefficient type:", type(v_i["e1"]).__name__)


float64: e1 + 2 e2 | coefficient type: float
int64  : e1 + 2 e2 | coefficient type: int


## 3. Creating multivectors

**Calling the algebra** (or, equivalently, `alg.multivector(...)`) builds an `MV`.
Five input forms are accepted. Here they all produce the same multivector
$1 + 2 e_1 - 3 e_2 + 4 e_{12}$:


In [5]:
# 1. Zero multivector
zero = alg()

# 2. Dict with integer keys — raw blade bitmasks (bit k = basis vector e_{k+1})
d_bitmask = alg({0: 1.0, 1: 2.0, 2: -3.0, 3: 4.0})

# 3. Dict with string keys — blade names ("s" is the scalar)
d_str = alg({"s": 1.0, "e1": 2.0, "e2": -3.0, "e12": 4.0})

# 4. Dict with tuple keys — 1-based vector-index tuples; (0,) or () for scalar
d_tuple = alg({(0,): 1.0, (1,): 2.0, (2,): -3.0, (1, 2): 4.0})

# 5. String expression — a sum of signed terms
d_expr = alg("1 + 2 e1 - 3 e2 + 4 e12")

# All five forms build the same blades (compare via to_dict())
for name, m in [("bitmask", d_bitmask), ("str-keys", d_str),
                ("tuple-keys", d_tuple), ("string", d_expr)]:
    print(f"{name:>10}: {m}")

print()
print("identical:", d_bitmask.to_dict() == d_str.to_dict()
                     == d_tuple.to_dict() == d_expr.to_dict())
print("zero     :", repr(zero))
print("type     :", type(d_expr).__name__)


   bitmask: 1 + 2 e1 - 3 e2 + 4 e12
  str-keys: 1 + 2 e1 - 3 e2 + 4 e12
tuple-keys: 1 + 2 e1 - 3 e2 + 4 e12
    string: 1 + 2 e1 - 3 e2 + 4 e12

identical: True
zero     : 0
type     : MV


## 4. Coefficient access & display

Blades are addressed by **name** (`"e1"`, `"e12"`, `"s"`, `"I"`) or by **raw bitmask**
(`1` → $e_1$, `3` → $e_{12}$, `0` → scalar). Reading an absent blade returns `0`:


In [6]:
mv = alg("3 e1 + 5 e12")

print("mv['e1']  =", mv["e1"])   # by blade name
print("mv['e12'] =", mv["e12"])
print("mv['e2']  =", mv["e2"])   # absent blade -> 0
print("mv[1]     =", mv[1])      # bitmask 1 -> e1

# Writing works the same way
mv["e2"] = -7.0
mv[3]    = 2.0                   # bitmask 3 = 0b11 -> e12
print()
print("after writing:", mv)


mv['e1']  = 3.0
mv['e12'] = 5.0
mv['e2']  = 0.0
mv[1]     = 3.0

after writing: 3 e1 - 7 e2 + 2 e12


In [7]:
# repr() gives a compact expression
print("repr    :", repr(mv))

# show() prints a labelled line (fmt controls the coefficient format)
mv.show("mv")
mv.show("mv ('.2f')", fmt=".2f")

# to_dict() returns {blade_name: coefficient}
print("to_dict :", mv.to_dict())


repr    : 3 e1 - 7 e2 + 2 e12


mv: 3 e1 - 7 e2 + 2 e12

mv ('.2f'): 3.00 e1 - 7.00 e2 + 2.00 e12

to_dict : {'e1': 3.0, 'e2': -7.0, 'e12': 2.0}


## 5. Arithmetic

Multivectors support `+`, `-`, and scalar `*` / `/`. A plain `int` or `float` on either
side is automatically promoted to a scalar multivector:


In [8]:
u = alg("e1 + 2 e2")
w = alg("3 e1 - e2")

print("u + w  =", u + w)
print("u - w  =", u - w)
print("-u     =", -u)
print("3 * u  =", 3 * u)      # scalar left
print("u * 3  =", u * 3)      # scalar right
print("u / 4  =", u / 4)
print("u + 1  =", u + 1)      # scalar promoted to a scalar MV
print("1 - u  =", 1 - u)


u + w  = 4 e1 + e2
u - w  = -2 e1 + 3 e2
-u     = -e1 - 2 e2
3 * u  = 3 e1 + 6 e2
u * 3  = 3 e1 + 6 e2
u / 4  = 0.25 e1 + 0.5 e2
u + 1  = 1 + e1 + 2 e2
1 - u  = 1 - e1 - 2 e2


## 6. The three products

Three products lie at the heart of geometric algebra. In 3D Euclidean space they act on
vectors as follows:

- **Geometric product** `a * b` (or `a.gp(b)`): the fundamental product. For vectors it
  combines the inner and outer products: $ab = a \cdot b + a \wedge b$.
- **Outer product** `a ^ b` (or `a.op(b)`): the anti-symmetric part — the oriented
  *area* spanned by $a$ and $b$ (a bivector).
- **Inner product** `a | b` (or `a.ip(b)`): the symmetric (metric) part — the familiar
  dot product for vectors.

For orthogonal unit vectors, `e1 * e2` is purely the bivector `e12`; for equal vectors,
`e1 * e1` is purely the scalar `1`:


In [9]:
e1 = alg("e1")
e2 = alg("e2")

print("e1 * e2 =", e1 * e2)      # geometric product -> bivector e12
print("e1 ^ e2 =", e1 ^ e2)      # outer product   -> same, for orthogonal vectors
print("e1 | e1 =", e1 | e1)      # inner product   -> scalar 1 (the dot product)
print("e1 | e2 =", e1 | e2)      # orthogonal      -> 0

# For non-orthogonal vectors the GP has two grades: scalar + bivector
a = alg("e1")
b = alg("e1 + e2")
print()
print("a * b         =", a * b)
print("scalar part   :", (a * b).grade(0))
print("bivector part :", (a * b).grade(2))


e1 * e2 = e12
e1 ^ e2 = e12
e1 | e1 = 1
e1 | e2 = 0

a * b         = 1 + e12
scalar part   : 1
bivector part : e12


## 7. Reverse, involutions, and the inverse

The **reverse** $\tilde{A}$ reverses the order of basis vectors inside every blade,
negating blades of grade $k$ where $k(k-1)/2$ is odd (i.e. grades 2 and 3 mod 4).
It is written `~a` or `a.rev()`.

> **⚠ Gotcha:** in *pytanga*, `~a` is the **reverse**, *not* the multiplicative
> inverse. Use `a.inv()` for the inverse. (Some GA libraries use `~` for the inverse —
> here it follows the "tilde = reverse" convention.)

The **Clifford conjugate** `conj()` applies the reverse plus a metric-dependent sign.
The **grade involution** `grade_involution()` negates every odd-grade part, and the
**grade conjugate** `grade_conj()` is the grade involution followed by the reverse.


In [10]:
m = alg("1 + 2 e1 + 3 e12")

print("m                =", m)
print("~m (reverse)     =", ~m)
print("m.rev()          =", m.rev())
print("m.conj()         =", m.conj())
print("grade_involution =", m.grade_involution())
print("grade_conj       =", m.grade_conj())


m                = 1 + 2 e1 + 3 e12
~m (reverse)     = 1 + 2 e1 - 3 e12
m.rev()          = 1 + 2 e1 - 3 e12
m.conj()         = 1 + 2 e1 - 3 e12
grade_involution = 1 - 2 e1 + 3 e12
grade_conj       = 1 - 2 e1 - 3 e12


In [11]:
# The multiplicative inverse: a * a.inv() == 1
v = alg("2 e1 + 3 e2")          # a plain vector

print("v           =", v)
print("v.inv()     =", v.inv())
print("v * v.inv() =", v * v.inv())   # -> 1
print("v * ~v      =", v * ~v)        # ~v == v for a vector; gives |v|^2 = 13


v           = 2 e1 + 3 e2
v.inv()     = 0.1538 e1 + 0.2308 e2
v * v.inv() = 1
v * ~v      = 13


## 8. Grade extraction & blade iteration

A general multivector is a sum of parts of different *grades* (scalar = grade 0,
vector = grade 1, bivector = grade 2, …). You can extract any grade, split into
even/odd parts, list the present grades, and decompose into individual blades:


In [12]:
x = alg("1 + 2 e1 + 3 e2 + 4 e12 + 5 I")   # scalar + vector + bivector + pseudoscalar

print("x            =", x)
print("x.grades     =", x.grades)           # grades with non-zero coefficients
print("grade 0      =", x.grade(0))
print("grade 1      =", x.grade(1))
print("grade 2      =", x.grade(2))
print("grade [1, 2] =", x.grade([1, 2]))    # sum of several grades
print("even part    =", x.even())
print("odd part     =", x.odd())


x            = 1 + 2 e1 + 3 e2 + 4 e12 + 5 I
x.grades     = [0, 1, 2, 3]
grade 0      = 1
grade 1      = 2 e1 + 3 e2
grade 2      = 4 e12
grade [1, 2] = 2 e1 + 3 e2 + 4 e12
even part    = 1 + 4 e12
odd part     = 2 e1 + 3 e2 + 5 I


In [13]:
# Decompose into a list of single-blade multivectors
print("components:")
for blade in x.components():
    print("   ", blade)

# Coefficients in canonical blade order (s, e1, e2, e3, e12, e13, e23, I)
print()
print("all blade coefficients:", x.blade_coefs())

# Grade-k coefficients only (e1, e2, e3 for k=1)
print("grade-1 coefficients :", x.get_coefs(1))


components:
    1
    2 e1
    3 e2
    4 e12
    5 I

all blade coefficients: [1.0, 2.0, 3.0, 0.0, 4.0, 0.0, 0.0, 5.0]
grade-1 coefficients : [2.0, 3.0, 0.0]


## 9. `prune()` and precision

Floating-point arithmetic often leaves tiny numerical residue (e.g. $10^{-16}$) on
blades that should be zero. The algebra has a **precision** threshold (default
`1e-10`); `prune()` removes every coefficient below it. You can also pass an explicit
tolerance:


In [14]:
noisy = alg({1: 1e-6, 2: 1e-12, 3: 2.0})   # e1 (1e-6), e2 (1e-12), e12 (2.0)

print("precision    :", alg.precision)
print("before       :", noisy.to_dict())

noisy.prune()                                # tol = alg.precision (1e-10)
print("prune()      :", noisy.to_dict())     # drops 1e-12, keeps 1e-6

noisy2 = alg({1: 1e-6, 2: 1e-12, 3: 2.0})
noisy2.prune(1e-4)                           # coarser tolerance
print("prune(1e-4)  :", noisy2.to_dict())    # drops 1e-6 and 1e-12, keeps 2.0


precision    : 1e-10
before       : {'e1': 1e-06, 'e2': 1e-12, 'e12': 2.0}
prune()      : {'e1': 1e-06, 'e12': 2.0}
prune(1e-4)  : {'e12': 2.0}


## 10. Summary & next steps

You now know the core of *pytanga*:

| Concept | API |
|---|---|
| Build an algebra | `Algebra(dim, sig, dtype)` |
| Create a multivector | `alg(...)` / `alg.multivector(...)` |
| Products | `*` / `^` / `\|` (`gp` / `op` / `ip`) |
| Reverse / inverse | `~a` (reverse) / `a.inv()` (inverse) |
| Grade extraction | `a.grade(k)`, `a.even()`, `a.odd()` |
| Blade iteration | `a.components()`, `a.blade_coefs()` |
| Cleanup | `a.prune()` |

**Where to go next:**

- [**03 · The Eight Basis Classes**](../03_basis_classes/) — prebuilt algebras with
  named blades (`BasisE3`, `BasisPGA3`, …); they pre-define those blades and are
  required by the `Geometry` submodule.
- [**01 · Algebra Quick Tour**](../01_quick_tour/) — a big-picture overview of the whole
  algebra side, if you want the lay of the land first.

> This tutorial used the generic `Algebra` constructor, which builds any algebra
> from scratch. The basis classes (next tutorial) pre-define named blades and are
> required when you use the `Geometry` submodule.